# 04 - Modelo de Referência e Primeiros Modelos

Começo a modelagem. Neste notebook:

- monto um modelo de referência bem simples (baseline);
- treino os dois primeiros modelos de verdade: regressão logística e árvore
  de decisão;
- comparo com validação cruzada no tempo, olhando métricas que fazem sentido
  para classes desbalanceadas (não só a acurácia).

A comparação com modelos mais fortes fica no notebook 05.

## 1. Preparação

In [1]:
import sys
sys.path.append("..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from src.preprocessing import (preparar, separar_treino_teste,
                               VARIAVEIS_NUMERICAS, VARIAVEIS_CATEGORICAS, ALVO)

pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [2]:
df = preparar()
treino, teste = separar_treino_teste(df)

variaveis = VARIAVEIS_NUMERICAS + VARIAVEIS_CATEGORICAS
X_treino, y_treino = treino[variaveis], treino[ALVO]
X_teste, y_teste = teste[variaveis], teste[ALVO]

print("treino:", X_treino.shape, " teste:", X_teste.shape)

treino: (95367, 33)  teste: (23842, 33)


## 2. Pré-processamento

Um `ColumnTransformer`: nas numéricas preencho ausentes com a mediana e
padronizo; nas categóricas preencho com o valor mais frequente e aplico
one-hot. Vai dentro do pipeline de cada modelo, então é ajustado só com o
treino de cada divisão da validação cruzada.

In [3]:
preproc = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), VARIAVEIS_NUMERICAS),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), VARIAVEIS_CATEGORICAS),
])

## 3. Validação cruzada no tempo

Uso `TimeSeriesSplit`: cada divisão treina num período e valida no período
seguinte. É a mesma ideia de K-Fold, mas respeitando a ordem do tempo. Uso o
mesmo `cv` para todos os modelos, então a comparação é justa.

In [4]:
cv = TimeSeriesSplit(n_splits=5)
metricas = ["accuracy", "recall", "precision", "f1", "roc_auc", "average_precision"]

modelos = {
    "referência (dummy)": DummyClassifier(strategy="most_frequent"),
    "regressão logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "árvore de decisão": DecisionTreeClassifier(class_weight="balanced", random_state=42),
}

linhas = []
for nome, modelo in modelos.items():
    pipe = Pipeline([("prep", preproc), ("modelo", modelo)])
    r = cross_validate(pipe, X_treino, y_treino, cv=cv, scoring=metricas)
    linhas.append({
        "modelo": nome,
        "accuracy": r["test_accuracy"].mean(),
        "recall": r["test_recall"].mean(),
        "precision": r["test_precision"].mean(),
        "f1": r["test_f1"].mean(),
        "roc_auc": r["test_roc_auc"].mean(),
        "pr_auc": r["test_average_precision"].mean(),
    })

tabela = pd.DataFrame(linhas).set_index("modelo").round(3)
tabela

,accuracy,recall,precision,f1,roc_auc,pr_auc
modelo,,,,,,
referência (dummy),0.647,0.000,0.000,0.000,0.500,0.353
regressão logística,0.804,0.622,0.783,0.689,0.872,0.819
árvore de decisão,0.774,0.549,0.744,0.626,0.725,0.572


O modelo de referência prevê sempre a classe mais comum. Tem uma acurácia
que parece razoável só porque a maior parte das reservas não é cancelada, mas
recall 0 (não identifica nenhum cancelamento) e ROC-AUC 0,5. É a referência
que eu queria: qualquer modelo útil precisa ficar claramente acima disso.

A regressão logística e a árvore já ficam bem acima. A árvore sem poda acerta
a classe, mas costuma dar probabilidades piores (Brier maior), o que dá para
ver melhor no notebook 05.

## 4. O que levo para o próximo notebook

- a acurácia sozinha engana neste problema; vou olhar recall, F1, ROC-AUC e
  PR-AUC;
- a regressão logística é uma referência simples e forte, o modelo a ser
  superado;
- o pré-processamento (`preproc`) e a validação cruzada no tempo (`cv`) valem
  para todos os modelos daqui em diante;
- no notebook 05 comparo modelos mais fortes (random forest, gradient
  boosting, XGBoost e uma rede neural) com o mesmo esquema.